# Day 6 - Event Stream Freshness Proof

Business Screnario: logistics control tower delay response
Goal: simulate events, compute lag, flga late/duplicate records, and feed a governed agent signal.

In [5]:
from datetime import datetime, timedelta, timezone
import pandas as pd 

base = datetime(2026, 8, 3, 10, 0, tzinfo=timezone.utc)

events = pd.DataFrame([
    { 
        "event_id": "EVT-001", 
        "shipment_id": "SHIP_1001", 
        "event_type": "pickup", 
        "event_time": base,
        "processing_time": base + timedelta(seconds=40),
        "status": "picked_up",
    },
    { 
        "event_id": "EVT-002", 
        "shipment_id": "SHIP_1001", 
        "event_type": "delay", 
        "event_time": base + timedelta(minutes=8),
        "processing_time": base + timedelta(minutes=10, seconds=20),
        "status": "weather_delay",
    },
    { 
        "event_id": "EVT-002", 
        "shipment_id": "SHIP_1001", 
        "event_type": "delay", 
        "event_time": base + timedelta(minutes=8),
        "processing_time": base + timedelta(minutes=10, seconds=50),
        "status": "weather_delay",
    },
    { 
        "event_id": "EVT-003", 
        "shipment_id": "SHIP_1002", 
        "event_type": "location_update", 
        "event_time": base - timedelta(hours=2),
        "processing_time": base + timedelta(minutes=11),
        "status": "late_arrival",
    },
    { 
        "event_id": None, 
        "shipment_id": "SHIP_1003", 
        "event_type": "delay", 
        "event_time": base + timedelta(minutes=12),
        "processing_time": base + timedelta(minutes=12, seconds=15),
        "status": "missing_event_id",
    },
])

events["lag_seconds"] = (events["processing_time"] - events["event_time"]).dt.total_seconds().astype(int)
events["is_duplicate"] = events.duplicated("event_id", keep="first") & events["event_id"].notna()
events["is_late"] = events["lag_seconds"] > 300
events["contract_valid"] = events[["event_id", "shipment_id", "event_type", "event_time", "processing_time"]].notna().all(axis=1)

gold = events[events["contract_valid"] & ~events["is_duplicate"]].copy()
dead_letter = events[~events["contract_valid"] | events["is_duplicate"]].copy()

slo = {
    "valid_events": int(gold.shape[0]),
    "dead_letter_events": int(dead_letter.shape[0]),
    "fresh_within_5m": int((gold["lag_seconds"] <= 300).sum()),
    "freshness_percent":  round(float((gold["lag_seconds"] <= 300).mean() * 100), 2),
    "worst_lag_seconds": int(gold["lag_seconds"].max()),
}

print("GOLD TABLE")
print(gold[["event_id", "shipment_id", "event_type", "lag_seconds", "is_late", "status" ]].to_string(index=False))
print("\nDEAD LETTER")
print(dead_letter[["event_id", "shipment_id", "event_type", "contract_valid", "is_duplicate", "status" ]].to_string(index=False))
print("\nSLO")
print(slo)



GOLD TABLE
event_id shipment_id      event_type  lag_seconds  is_late        status
 EVT-001   SHIP_1001          pickup           40    False     picked_up
 EVT-002   SHIP_1001           delay          140    False weather_delay
 EVT-003   SHIP_1002 location_update         7860     True  late_arrival

DEAD LETTER
event_id shipment_id event_type  contract_valid  is_duplicate           status
 EVT-002   SHIP_1001      delay            True          True    weather_delay
     NaN   SHIP_1003      delay           False         False missing_event_id

SLO
{'valid_events': 3, 'dead_letter_events': 2, 'fresh_within_5m': 2, 'freshness_percent': 66.67, 'worst_lag_seconds': 7860}


In [8]:
agent_signal = []
for _, row in gold.iterrows():
    if row["event_type"] == "delay" and row["lag_seconds"] <= 300:
        agent_signal.append({
            "shipment_id": row["shipment_id"],
            "signal": "delay_within_freshness_slo",
            "recommendation": "draft customer update and dispatcher review task",
            "auto_execute": False,
        })
agent_signal

[{'shipment_id': 'SHIP_1001',
  'signal': 'delay_within_freshness_slo',
  'recommendation': 'draft customer update and dispatcher review task',
  'auto_execute': False}]